# Base-models CV (text / whisper / wavlm — RF & XGB, no fusions)

Goal: get **honest, candidate-level** numbers for every base model in a way that does not produce the jagged precision jumps you saw with single-batch CV. The fix is the **mixed CV pool** below: 20% of audios4 + 40% of audios5 are held out together for threshold tuning, and the remaining 60% of audios5 is the **untouched test**.

## Splits (always candidate-level, never file-level)
Per fold (`N_FOLDS=5` random candidate splits with different seeds — Monte-Carlo CV):
- **Train**: full audios2 + **80%** of audios4 candidates
- **CV**: **20%** of audios4 candidates + **`CV_A5_FRAC` (default 0.40)** of audios5 candidates → threshold tuning surface
- **Test**: remaining **`1 − CV_A5_FRAC` (default 0.60)** of audios5 candidates → never seen during training or threshold pick
- Same candidate never appears on both sides of any split.

Why MC-CV instead of partitioned K-fold: the user's CV-vs-test ratio (40/60 on a5) doesn't divide cleanly into 5 disjoint folds. Random candidate-level splits with rotated seeds give the same per-fold variance signal without forcing the ratio to equal `1/K`.

## Duration filter
`speaking_time_s ≥ MIN_SPEAKING_S` (default 30) applied to **every** batch on **every** side (train, CV, test). Reads from `checkpoints_honest_eval/{batch}_durations.csv`. Same filter on the test side as on training, per the user's note that <30s audios add noise.

## Models (9, no fusions)
| name | head | features |
|---|---|---|
| `text_top10_xgb`  | XGB | top-10 text features by audios2-only XGB importance |
| `text_top15_xgb`  | XGB | top-15 |
| `text_top20_xgb`  | XGB | top-20 |
| `text_stylo_xgb`  | XGB | 15 stylometric features |
| `text_all_xgb`    | XGB | all 55 text features |
| `whisper_wp_rf`   | RF  | frozen Whisper-medium whole-pool (1024d) |
| `whisper_wp_xgb`  | XGB | same features |
| `wavlm_wp_rf`     | RF  | frozen WavLM-base-plus **pretrained** whole-pool (768d) |
| `wavlm_wp_xgb`    | XGB | same features |

All heads use deployment-prior reweighting: XGB `scale_pos_weight=SPW_DEPLOY≈4.88`, RF `class_weight={0:1, 1:SPW_DEPLOY}` — so RF and XGB metrics are directly comparable.

## Metrics per (model, fold, strategy)
Strategies: best-CV-F1 (`F1`) and CV-precision-floor (`P80, P85, P90, P95`).
- For `F1`: `thr` (max CV F1), `cv_f1, cv_prec, cv_rec, te_f1, te_prec, te_rec, gap_f1 = cv_f1 − te_f1`.
- For each `P{X}`: `thr` (max CV recall s.t. CV prec ≥ X%), `cv_rec, cv_prec, te_prec, te_rec, te_f1, gap_rec`.

## Outputs
- `checkpoints_basemodels/per_fold.csv` — every fold × model × strategy row (raw)
- `checkpoints_basemodels/summary_avg.csv` — per-model mean across folds with std on the headline cols
- Notebook display: avg table only (per-fold lives in CSV — keeps the notebook readable)

In [ ]:
# === 0. Setup ===
from pathlib import Path
import re, warnings, time
import numpy as np
import pandas as pd

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

warnings.filterwarnings('ignore')

NB_DIR        = Path('.').resolve()
SAVE_DIR      = NB_DIR / 'checkpoints_basemodels'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
DURATIONS_DIR = NB_DIR / 'checkpoints_honest_eval'   # {batch}_durations.csv lives here

# Split protocol
MIN_SPEAKING_S = 30
N_FOLDS        = 5
CV_A4_FRAC     = 0.20   # share of audios4 candidates held out for CV each fold
CV_A5_FRAC     = 0.40   # share of audios5 candidates held out for CV each fold (rest = test)
RANDOM_SEED    = 42

# Deployment-prior loss reweighting (audios4/5 ~17% positive)
DEPLOY_POS_RATE = 0.17
SPW_DEPLOY      = (1.0 - DEPLOY_POS_RATE) / DEPLOY_POS_RATE

# Threshold-pick strategies
STRATEGIES = ['F1','P80','P85','P90','P95']
PREC_FLOOR = {'P80':0.80,'P85':0.85,'P90':0.90,'P95':0.95}

LABEL_MAP = {
    'read':1,'cheating':1,'reading':1,'scripted':1,'yes':1,'1':1,1:1,
    'spontaneous':0,'not cheating':0,'not_cheating':0,'no':0,'0':0,0:0,'genuine':0,
}

# 55 text features (matches text_cheating_detection.ipynb / fusion notebooks)
ALL_TEXT_FEATURES = [
    # disfluency (6)
    'filler_rate','filler_count','repetition_rate','repair_rate','discourse_marker_rate','hedge_rate',
    # stylometric (15)
    'ttr','mattr','mtld','complex_word_rate','avg_word_length','n_words','n_unique_words',
    'avg_sentence_length','std_sentence_length','fragment_rate','n_sentences','self_ref_rate',
    'noun_rate','verb_rate','adj_rate',
    # pause (15)
    'pause_mean','pause_std','pause_median','pause_skew','long_pause_rate','pause_ratio','n_pauses',
    'pause_regularity','pause_before_content_ratio','pause_before_function_ratio','mid_phrase_pause_rate',
    'words_per_sec','articulation_rate','initial_pause','longest_pause',
    # suspicious (2)
    'suspicious_gap_count','suspicious_gap_ratio',
    # formal_ai (4)
    'formal_transition_count','formal_transition_rate','ai_phrase_count','ai_phrase_rate',
    # prosodic (8)
    'f0_mean','f0_std','f0_range','f0_skew','f0_slope','energy_mean','energy_std','speaking_rate_std',
    # voice_q (3)
    'jitter_local','shimmer_local','hnr_mean',
    # perplexity (2)
    'mean_perplexity','burstiness',
]
STYLO_FEATS = ['ttr','mattr','mtld','complex_word_rate','avg_word_length','n_words','n_unique_words',
               'avg_sentence_length','std_sentence_length','fragment_rate','n_sentences','self_ref_rate',
               'noun_rate','verb_rate','adj_rate']

# Subset used for top-N ranking (excludes prosodic/voice_q/perplexity to match fusion_text_wavlm convention)
TEXT_RANK_COLS = [f for f in ALL_TEXT_FEATURES
                  if f not in ('f0_mean','f0_std','f0_range','f0_skew','f0_slope',
                               'energy_mean','energy_std','speaking_rate_std',
                               'jitter_local','shimmer_local','hnr_mean',
                               'mean_perplexity','burstiness')]

BATCHES = ['audios2','audios4','audios5']
print(f'NB_DIR={NB_DIR}\nSAVE_DIR={SAVE_DIR}')
print(f'MIN_SPEAKING_S={MIN_SPEAKING_S}  N_FOLDS={N_FOLDS}  CV_A4_FRAC={CV_A4_FRAC}  CV_A5_FRAC={CV_A5_FRAC}')
print(f'SPW_DEPLOY={SPW_DEPLOY:.2f}')

In [ ]:
# === 1. Data loading + duration filter + candidate_id ===
WAVLM_WHOLE_CANDIDATES = lambda n: [f'{n}_wavlm_whole.csv', f'{n}_whole_pretrained.csv']
# (To switch to finetuned: change to [f'{n}_whole_finetuned.csv'].)

def _first_existing(cands):
    for c in cands:
        p = NB_DIR / c
        if p.exists(): return p
    raise FileNotFoundError(f'None of {cands} exist under {NB_DIR}')

def load_gt(name):
    gt = pd.read_csv(NB_DIR / f'{name}GT.csv')
    fn_col  = next(c for c in gt.columns if c.lower() in ('filename','file','name'))
    lbl_col = next(c for c in gt.columns if c.lower() in ('label','class','cheating','gt','label_int','ground_truth'))
    gt = gt.rename(columns={fn_col:'filename', lbl_col:'label_raw'})
    gt['label_int'] = gt['label_raw'].map(
        lambda x: LABEL_MAP.get(x, LABEL_MAP.get(str(x).lower().strip(), -1)))
    return gt[gt['label_int'].isin([0,1])][['filename','label_int']]

def load_durations(name):
    p = DURATIONS_DIR / f'{name}_durations.csv'
    if not p.exists():
        print(f'  WARN: {p} missing — duration filter will be a no-op for {name}')
        return None
    return pd.read_csv(p)[['filename','speaking_time_s']]

def load_folder(name):
    gt   = load_gt(name)
    text = pd.read_csv(NB_DIR / f'{name}_features.csv')
    df   = gt.merge(text, on='filename', how='inner')
    wp   = pd.read_csv(_first_existing(WAVLM_WHOLE_CANDIDATES(name)))
    df   = df.merge(wp, on='filename', how='inner')
    wh   = pd.read_csv(NB_DIR / f'{name}_whisper_whole.csv')
    df   = df.merge(wh, on='filename', how='inner')
    df['batch'] = name
    dur = load_durations(name)
    if dur is not None:
        df = df.merge(dur, on='filename', how='left')
    return df

def filter_by_duration(df, min_s):
    if min_s <= 0 or 'speaking_time_s' not in df.columns: return df
    keep = (df['speaking_time_s'] >= min_s) | df['speaking_time_s'].isna()
    return df[keep].reset_index(drop=True)

_RE_CAND = re.compile(r'^(.+)_(\d{1,3})\.[a-zA-Z0-9]+$')
def attach_candidate_id(df):
    df = df.copy()
    df['candidate_id'] = df['filename'].astype(str).map(
        lambda f: (_RE_CAND.match(f).group(1) if _RE_CAND.match(f) else None))
    return df

batches_full = {b: attach_candidate_id(load_folder(b))            for b in BATCHES}
batches      = {b: filter_by_duration(batches_full[b], MIN_SPEAKING_S) for b in BATCHES}

print('=== Per-batch counts (raw -> filtered at >= {}s) ==='.format(MIN_SPEAKING_S))
for b in BATCHES:
    f0, f1 = batches_full[b], batches[b]
    y0, y1 = f0['label_int'].values, f1['label_int'].values
    print(f'  {b}:  rows {len(f0):4d} -> {len(f1):4d}   '
          f'cheat {int((y0==1).sum()):3d}->{int((y1==1).sum()):3d}   '
          f'honest {int((y0==0).sum()):3d}->{int((y1==0).sum()):3d}   '
          f'candidates {f0["candidate_id"].nunique()}->{f1["candidate_id"].nunique()}')

In [ ]:
# === 2. Feature columns + top-N text feature ranking on audios2 only ===
first = batches['audios2']
WH_COLS    = [c for c in first.columns if c.startswith('whisper_')]
WP_COLS    = [c for c in first.columns if c.startswith('wavlm_')
              and not c.startswith('wavlm_mean_') and not c.startswith('wavlm_std_')]
TEXT_ALL   = [c for c in ALL_TEXT_FEATURES if c in first.columns]
TEXT_STYLO = [c for c in STYLO_FEATS       if c in first.columns]
TEXT_RANK  = [c for c in TEXT_RANK_COLS    if c in first.columns]

# Rank top-N text features on audios2 only — no a4/a5 leakage
_X = first[TEXT_RANK].fillna(0).values
_y = first['label_int'].values
_sc = StandardScaler().fit(_X)
_rkr = xgb.XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
    random_state=RANDOM_SEED)
_rkr.fit(_sc.transform(_X), _y)
_imp = pd.Series(_rkr.feature_importances_, index=TEXT_RANK).sort_values(ascending=False)
TEXT_TOP10 = _imp.head(10).index.tolist()
TEXT_TOP15 = _imp.head(15).index.tolist()
TEXT_TOP20 = _imp.head(20).index.tolist()

print(f'  whisper_wp d={len(WH_COLS)}  wavlm_wp d={len(WP_COLS)}  '
      f'text_all d={len(TEXT_ALL)}  text_stylo d={len(TEXT_STYLO)}  text_rank d={len(TEXT_RANK)}')
print('\nTop-20 text features (XGB importance, audios2 only):')
for i, f in enumerate(TEXT_TOP20, 1):
    mark = '   <- top-10' if i == 10 else ('   <- top-15' if i == 15 else ('   <- top-20' if i == 20 else ''))
    print(f'  {i:2d}. {f:32s}  imp={_imp[f]:.4f}{mark}')

In [ ]:
# === 3. Model factories + registry ===
def make_xgb(n_feats, seed=RANDOM_SEED):
    cs = 0.3 if n_feats > 500 else 0.8
    return xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=cs, min_child_weight=3,
        scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
        random_state=seed)

def make_rf(n_feats, seed=RANDOM_SEED):
    return RandomForestClassifier(
        n_estimators=500, max_depth=8, min_samples_leaf=3,
        class_weight={0:1.0, 1:float(SPW_DEPLOY)},
        n_jobs=-1, random_state=seed)

def mk_X(cols): return lambda d: d[cols].fillna(0).values

BASE_REGISTRY = {
    'text_top10_xgb':  (mk_X(TEXT_TOP10), lambda s=RANDOM_SEED: make_xgb(len(TEXT_TOP10), s)),
    'text_top15_xgb':  (mk_X(TEXT_TOP15), lambda s=RANDOM_SEED: make_xgb(len(TEXT_TOP15), s)),
    'text_top20_xgb':  (mk_X(TEXT_TOP20), lambda s=RANDOM_SEED: make_xgb(len(TEXT_TOP20), s)),
    'text_stylo_xgb':  (mk_X(TEXT_STYLO), lambda s=RANDOM_SEED: make_xgb(len(TEXT_STYLO), s)),
    'text_all_xgb':    (mk_X(TEXT_ALL),   lambda s=RANDOM_SEED: make_xgb(len(TEXT_ALL),   s)),
    'whisper_wp_rf':   (mk_X(WH_COLS),    lambda s=RANDOM_SEED: make_rf (len(WH_COLS),    s)),
    'whisper_wp_xgb':  (mk_X(WH_COLS),    lambda s=RANDOM_SEED: make_xgb(len(WH_COLS),    s)),
    'wavlm_wp_rf':     (mk_X(WP_COLS),    lambda s=RANDOM_SEED: make_rf (len(WP_COLS),    s)),
    'wavlm_wp_xgb':    (mk_X(WP_COLS),    lambda s=RANDOM_SEED: make_xgb(len(WP_COLS),    s)),
}
MODELS = list(BASE_REGISTRY.keys())
print('Models:', MODELS)

In [ ]:
# === 4. Candidate-level fold split ===
def candidate_split(df, frac_holdout, seed):
    """Return (train_part, holdout_part) split at candidate level.
    Sampling is over candidates (not rows) so all of a candidate's audios go
    to the same side. Stratification is on candidate-level *any positive*
    so the small CV slice keeps the cheating/honest mix close to the batch's."""
    rng = np.random.default_rng(seed)
    cand_label = df.groupby('candidate_id')['label_int'].max()  # candidate is 'positive' if any audio is
    pos_cands = cand_label[cand_label==1].index.tolist()
    neg_cands = cand_label[cand_label==0].index.tolist()
    rng.shuffle(pos_cands); rng.shuffle(neg_cands)
    n_pos_h = max(1, int(round(len(pos_cands) * frac_holdout)))
    n_neg_h = max(1, int(round(len(neg_cands) * frac_holdout)))
    holdout = set(pos_cands[:n_pos_h] + neg_cands[:n_neg_h])
    train_part   = df[~df['candidate_id'].isin(holdout)].reset_index(drop=True)
    holdout_part = df[ df['candidate_id'].isin(holdout)].reset_index(drop=True)
    return train_part, holdout_part

def build_fold(fold_idx):
    seed = RANDOM_SEED + fold_idx
    a4_train, a4_cv = candidate_split(batches['audios4'], CV_A4_FRAC, seed=seed)
    a5_test,  a5_cv = candidate_split(batches['audios5'], CV_A5_FRAC, seed=seed + 1000)
    df_train = pd.concat([batches['audios2'], a4_train], ignore_index=True)
    df_cv    = pd.concat([a4_cv, a5_cv],                  ignore_index=True)
    df_test  = a5_test.reset_index(drop=True)
    # Defensive overlap check
    cv_cands   = set(df_cv['candidate_id'])
    test_cands = set(df_test['candidate_id'])
    train_cands = set(df_train['candidate_id'])
    assert not (cv_cands & test_cands),    f'fold {fold_idx}: CV/test candidate overlap'
    assert not (train_cands & test_cands), f'fold {fold_idx}: train/test candidate overlap'
    # train can overlap with cv at the *batch* level (a2 doesn't appear in cv) but never within a4/a5
    return df_train, df_cv, df_test

# Sanity print fold 0 sizes
_tr, _cv, _te = build_fold(0)
def _szline(tag, df):
    y = df['label_int'].values
    return (f'  {tag}: rows={len(df):4d}  cands={df["candidate_id"].nunique():3d}  '
            f'cheat={int((y==1).sum()):3d}  honest={int((y==0).sum()):3d}  '
            f'pos_rate={(y==1).mean():.2f}  '
            f'batches={dict(df.groupby("batch").size())}')
print('=== Fold 0 split (illustrative) ===')
print(_szline('TRAIN', _tr))
print(_szline('CV   ', _cv))
print(_szline('TEST ', _te))

In [ ]:
# === 5. Threshold pickers + metrics ===
def _best_f1_thr(p, y, grid=np.arange(0.20, 0.81, 0.01)):
    bt, bf = 0.5, -1.0
    for thr in grid:
        f = f1_score(y, (p >= thr).astype(int), zero_division=0)
        if f > bf: bf, bt = f, float(thr)
    return bt, bf

def _best_rec_at_prec(p, y, target, min_tp=3):
    best = None
    for thr in np.arange(0.99, 0.10, -0.01):
        pred = (p >= thr).astype(int)
        cm = confusion_matrix(y, pred, labels=[0,1])
        if cm[1,1] < min_tp: continue
        pp = precision_score(y, pred, zero_division=0)
        rr = recall_score(y, pred, zero_division=0)
        if pp >= target and (best is None or rr > best[1]):
            best = (float(thr), float(rr), float(pp))
    return best if best is not None else (None, None, None)

def _metrics_at(p, y, thr):
    if thr is None or len(p)==0:
        return dict(prec=np.nan, rec=np.nan, f1=np.nan, n=len(p))
    pred = (p >= thr).astype(int)
    return dict(
        prec=float(precision_score(y, pred, zero_division=0)),
        rec =float(recall_score(y, pred, zero_division=0)),
        f1  =float(f1_score(y, pred, zero_division=0)),
        n=int(len(y)),
    )

def fit_score(name, df_tr, df_cv, df_te, seed=RANDOM_SEED):
    X_fn, factory = BASE_REGISTRY[name]
    Xtr, ytr = X_fn(df_tr), df_tr['label_int'].values
    sc = StandardScaler().fit(Xtr)
    clf = factory(seed)
    clf.fit(sc.transform(Xtr), ytr)
    p_cv = clf.predict_proba(sc.transform(X_fn(df_cv)))[:, 1]
    p_te = clf.predict_proba(sc.transform(X_fn(df_te)))[:, 1]
    return p_cv, df_cv['label_int'].values, p_te, df_te['label_int'].values

def fold_rows(name, fold_idx, p_cv, y_cv, p_te, y_te):
    rows = []
    # F1 strategy
    thr, cv_f1 = _best_f1_thr(p_cv, y_cv)
    cv = _metrics_at(p_cv, y_cv, thr); te = _metrics_at(p_te, y_te, thr)
    rows.append({
        'model': name, 'fold': fold_idx, 'strategy': 'F1', 'thr': round(thr,3),
        'cv_f1': round(cv['f1'],4), 'cv_prec': round(cv['prec'],4), 'cv_rec': round(cv['rec'],4),
        'te_f1': round(te['f1'],4), 'te_prec': round(te['prec'],4), 'te_rec': round(te['rec'],4),
        'gap_primary': round(cv['f1'] - te['f1'],4),
        'cv_n': cv['n'], 'te_n': te['n'],
    })
    # Precision-floor strategies
    for s in [s for s in STRATEGIES if s != 'F1']:
        thr, cv_rec, cv_prec = _best_rec_at_prec(p_cv, y_cv, PREC_FLOOR[s])
        if thr is None:
            rows.append({
                'model': name, 'fold': fold_idx, 'strategy': s, 'thr': None,
                'cv_f1': np.nan, 'cv_prec': np.nan, 'cv_rec': np.nan,
                'te_f1': np.nan, 'te_prec': np.nan, 'te_rec': np.nan,
                'gap_primary': np.nan,
                'cv_n': len(y_cv), 'te_n': len(y_te),
            })
            continue
        cv = _metrics_at(p_cv, y_cv, thr); te = _metrics_at(p_te, y_te, thr)
        rows.append({
            'model': name, 'fold': fold_idx, 'strategy': s, 'thr': round(thr,3),
            'cv_f1': round(cv['f1'],4), 'cv_prec': round(cv['prec'],4), 'cv_rec': round(cv['rec'],4),
            'te_f1': round(te['f1'],4), 'te_prec': round(te['prec'],4), 'te_rec': round(te['rec'],4),
            'gap_primary': round(cv['rec'] - te['rec'],4),  # primary metric for P-strategies is recall
            'cv_n': cv['n'], 'te_n': te['n'],
        })
    return rows

print('Pickers + fit_score + fold_rows ready.')

In [ ]:
# === 6. Run all folds × all models ===
all_rows = []
t_total = time.time()
for fold_idx in range(N_FOLDS):
    df_tr, df_cv, df_te = build_fold(fold_idx)
    print(f'\n--- fold {fold_idx} ---  '
          f'train n={len(df_tr)} (+{int((df_tr["label_int"]==1).sum())})  '
          f'cv n={len(df_cv)} (+{int((df_cv["label_int"]==1).sum())})  '
          f'test n={len(df_te)} (+{int((df_te["label_int"]==1).sum())})')
    for name in MODELS:
        t0 = time.time()
        p_cv, y_cv, p_te, y_te = fit_score(name, df_tr, df_cv, df_te, seed=RANDOM_SEED + fold_idx)
        all_rows.extend(fold_rows(name, fold_idx, p_cv, y_cv, p_te, y_te))
        print(f'    {name:20s}  done in {time.time()-t0:5.1f}s')

per_fold = pd.DataFrame(all_rows)
per_fold.to_csv(SAVE_DIR / 'per_fold.csv', index=False)
print(f'\nAll done in {(time.time()-t_total)/60:.1f} min.  Saved per-fold CSV: {SAVE_DIR/"per_fold.csv"}  ({len(per_fold)} rows)')

In [ ]:
# === 7. Summary: per-model averages across folds (with std on key columns) ===
AGG_COLS = ['thr','cv_f1','cv_prec','cv_rec','te_f1','te_prec','te_rec','gap_primary']

def _agg(df, with_std=True):
    means = df[AGG_COLS].mean(numeric_only=True)
    stds  = df[AGG_COLS].std (numeric_only=True)
    out = {f'{c}': round(float(means[c]),4) if not np.isnan(means[c]) else np.nan for c in AGG_COLS}
    if with_std:
        for c in ['cv_f1','te_f1','gap_primary']:
            out[f'{c}_std'] = round(float(stds[c]),4) if not np.isnan(stds[c]) else np.nan
    out['n_folds'] = int(df['thr'].notna().sum() if df['thr'].dtype != 'O' else len(df))
    return out

summary_rows = []
for (m, s), grp in per_fold.groupby(['model','strategy']):
    row = {'model': m, 'strategy': s, **_agg(grp)}
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
# Order rows: model in MODELS order, strategy in STRATEGIES order
summary['_m_ord'] = summary['model'].map({m:i for i,m in enumerate(MODELS)})
summary['_s_ord'] = summary['strategy'].map({s:i for i,s in enumerate(STRATEGIES)})
summary = summary.sort_values(['_m_ord','_s_ord']).drop(columns=['_m_ord','_s_ord']).reset_index(drop=True)
summary.to_csv(SAVE_DIR / 'summary_avg.csv', index=False)

# Pretty per-strategy display
for s in STRATEGIES:
    sub = summary[summary['strategy']==s].copy()
    print('\n' + '='*120)
    if s == 'F1':
        print(f' STRATEGY = {s}  (threshold maximises CV F1; primary metric = F1; gap = cv_f1 − te_f1)')
    else:
        print(f' STRATEGY = {s}  (threshold = max CV recall s.t. CV prec ≥ {int(PREC_FLOOR[s]*100)}%; '
              f'primary metric = recall; gap = cv_rec − te_rec)')
    print('='*120)
    show = ['model','thr','cv_f1','cv_prec','cv_rec','te_f1','te_prec','te_rec','gap_primary','cv_f1_std','te_f1_std','n_folds']
    show = [c for c in show if c in sub.columns]
    with pd.option_context('display.max_columns', None, 'display.width', 220):
        print(sub[show].to_string(index=False, na_rep='  --'))

print(f'\nSaved summary: {SAVE_DIR/"summary_avg.csv"}')
print(f'Per-fold raw : {SAVE_DIR/"per_fold.csv"}  (group by model+strategy to see fold-level numbers)')

In [ ]:
# === 8. Per-fold detail — only printed for the F1 strategy (so the notebook stays scannable). ===
# Full per-fold detail for ALL strategies is in per_fold.csv.
f1_view = per_fold[per_fold['strategy']=='F1'].copy()
pivot_te = f1_view.pivot_table(index='model', columns='fold', values='te_f1').round(3)
pivot_cv = f1_view.pivot_table(index='model', columns='fold', values='cv_f1').round(3)
pivot_thr = f1_view.pivot_table(index='model', columns='fold', values='thr').round(3)
pivot_te.columns   = [f'te_f1_f{c}'  for c in pivot_te.columns]
pivot_cv.columns   = [f'cv_f1_f{c}'  for c in pivot_cv.columns]
pivot_thr.columns  = [f'thr_f{c}'    for c in pivot_thr.columns]
fold_detail = pd.concat([pivot_cv, pivot_te, pivot_thr], axis=1).reindex(MODELS)

print('='*120)
print(' Per-fold detail (F1 strategy only) — cv_f1, te_f1, threshold per fold')
print('='*120)
with pd.option_context('display.max_columns', None, 'display.width', 220):
    print(fold_detail.to_string(na_rep='  --'))
print('\n(For per-fold detail at P80/P85/P90/P95, read per_fold.csv and filter by strategy.)')

## How to read the output

**Headline display (cell 7) — one block per strategy.** Each block ranks the 9 base models on the same threshold-picking rule:
- `F1` = best CV F1; `cv_f1`, `te_f1`, `gap_primary = cv_f1 − te_f1`. A small `|gap_primary|` means the CV pool is calibrated — the threshold transfers honestly to test.
- `P80…P95` = max CV recall under a CV precision floor; `gap_primary = cv_rec − te_rec`. `te_prec` tells you whether the precision target actually held on test (if `te_prec < 0.80` for `P80`, the floor didn't transfer).
- `cv_f1_std`, `te_f1_std` show fold-to-fold instability — the precision "jagged jumps" you saw before should now be visibly smaller because the CV pool is bigger and mixed.

**Per-fold detail (cell 8)** — F1 strategy only, in the notebook. Look for any model whose `te_f1` swings wildly across folds (>0.10 spread) — that's the one whose threshold isn't stable enough to ship even if its average looks good.

**CSVs** (`checkpoints_basemodels/`):
- `per_fold.csv` — every fold × model × strategy row, all metrics.
- `summary_avg.csv` — one row per model × strategy with means + stds.

**Knobs to tweak before re-running:**
- `CV_A5_FRAC` (cell 0) — drop to 0.30 if the CV pool feels too rich; raise to 0.50 if the precision-floor strategies still show null thresholds (no row clears the floor).
- `MIN_SPEAKING_S` — raise to 45 if 30s still leaves noise; lower to 0 to compare with the unfiltered baseline.
- `N_FOLDS` — 5 is enough for the average; raise to 10 if the std columns still look noisy.

**To switch wavlm to finetuned**: change `WAVLM_WHOLE_CANDIDATES` in cell 1 to `[f'{n}_whole_finetuned.csv']` and re-run from cell 1. The model name `wavlm_wp_*` stays — same registry slots, finetuned features.